# EBAC - Regressão II - regressão múltipla

## Tarefa I

#### Previsão de renda II

Vamos continuar trabalhando com a base 'previsao_de_renda.csv', que é a base do seu próximo projeto. Vamos usar os recursos que vimos até aqui nesta base.

|variavel|descrição|
|-|-|
|data_ref                | Data de referência de coleta das variáveis |
|index                   | Código de identificação do cliente|
|sexo                    | Sexo do cliente|
|posse_de_veiculo        | Indica se o cliente possui veículo|
|posse_de_imovel         | Indica se o cliente possui imóvel|
|qtd_filhos              | Quantidade de filhos do cliente|
|tipo_renda              | Tipo de renda do cliente|
|educacao                | Grau de instrução do cliente|
|estado_civil            | Estado civil do cliente|
|tipo_residencia         | Tipo de residência do cliente (própria, alugada etc)|
|idade                   | Idade do cliente|
|tempo_emprego           | Tempo no emprego atual|
|qt_pessoas_residencia   | Quantidade de pessoas que moram na residência|
|renda                   | Renda em reais|

1. Separe a base em treinamento e teste (25% para teste, 75% para treinamento).
2. Rode uma regularização *ridge* com alpha = [0, 0.001, 0.005, 0.01, 0.05, 0.1] e avalie o $R^2$ na base de testes. Qual o melhor modelo?
3. Faça o mesmo que no passo 2, com uma regressão *LASSO*. Qual método chega a um melhor resultado?
4. Rode um modelo *stepwise*. Avalie o $R^2$ na vase de testes. Qual o melhor resultado?
5. Compare os parâmetros e avalie eventuais diferenças. Qual modelo você acha o melhor de todos?
6. Partindo dos modelos que você ajustou, tente melhorar o $R^2$ na base de testes. Use a criatividade, veja se consegue inserir alguma transformação ou combinação de variáveis.
7. Ajuste uma árvore de regressão e veja se consegue um $R^2$ melhor com ela.

In [3]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

df = pd.read_csv('previsao_de_renda.csv')

df['log_renda'] = np.log(df['renda'])

modelo = smf.ols(
    'log_renda ~ idade + tempo_emprego + qtd_filhos + '
    'C(sexo) + C(posse_de_veiculo) + C(posse_de_imovel) + '
    'C(tipo_renda) + C(educacao) + C(estado_civil) + C(tipo_residencia)',
    data=df
).fit()

print(modelo.summary())

                            OLS Regression Results                            
Dep. Variable:              log_renda   R-squared:                       0.357
Model:                            OLS   Adj. R-squared:                  0.356
Method:                 Least Squares   F-statistic:                     299.5
Date:                Wed, 06 May 2026   Prob (F-statistic):               0.00
Time:                        15:11:20   Log-Likelihood:                -13571.
No. Observations:               12427   AIC:                         2.719e+04
Df Residuals:                   12403   BIC:                         2.737e+04
Df Model:                          23                                         
Covariance Type:            nonrobust                                         
                                          coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------
In

In [15]:
# 1)
# 1) Limpar dados primeiro
df = df.dropna()

# 2) Criar variáveis
X = df.drop(columns=['renda', 'log_renda'])
y = df['log_renda']

# 3) Dummies
X = pd.get_dummies(X, drop_first=True)

# 4) Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

In [21]:
# 2) Ridge Regression
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

alphas = [ 0.001, 0.005, 0.01, 0.05, 0.1]

resultados_ridge = []

for alpha in alphas:
    modelo = Ridge(alpha=alpha)
    modelo.fit(X_train, y_train)
    
    y_pred = modelo.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    
    resultados_ridge.append((alpha, r2))

# Resultado
for alpha, r2 in resultados_ridge:
    print(f'alpha: {alpha} → R²: {r2:.4f}')

alpha: 0.001 → R²: 0.3628
alpha: 0.005 → R²: 0.3628
alpha: 0.01 → R²: 0.3628
alpha: 0.05 → R²: 0.3628
alpha: 0.1 → R²: 0.3628


In [23]:
# 3) LASSO
from sklearn.linear_model import Lasso

resultados_lasso = []

for alpha in alphas:
    modelo = Lasso(alpha=alpha)
    modelo.fit(X_train, y_train)
    
    y_pred = modelo.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    
    resultados_lasso.append((alpha, r2))

for alpha, r2 in resultados_lasso:
    print(f'alpha: {alpha} → R²: {r2:.4f}')

alpha: 0.001 → R²: 0.3647
alpha: 0.005 → R²: 0.3654
alpha: 0.01 → R²: 0.3638
alpha: 0.05 → R²: 0.3380
alpha: 0.1 → R²: 0.2912


In [27]:
import statsmodels.api as sm

# adicionar constante + garantir tipo numérico
X_train_sm = sm.add_constant(X_train).astype(float)

modelo_step = sm.OLS(y_train, X_train_sm).fit()

# previsões
X_test_sm = sm.add_constant(X_test).astype(float)
y_pred = modelo_step.predict(X_test_sm)

from sklearn.metrics import r2_score
r2_step = r2_score(y_test, y_pred)

print("R² Stepwise:", r2_step)

R² Stepwise: 0.3627880576889143


In [32]:
# 5) Comparação dos modelos
#Comparando os modelos:

print("Melhor Ridge:", max(resultados_ridge, key=lambda x: x[1]))
print("Melhor Lasso:", max(resultados_lasso, key=lambda x: x[1]))
print("Stepwise:", r2_step)

Melhor Ridge: (0.1, 0.3628303943819764)
Melhor Lasso: (0.005, 0.3653721397767047)
Stepwise: 0.3627880576889143


In [36]:
#  6) Melhorar o modelo
df['idade_2'] = df['idade']**2
df['tempo_emprego_log'] = np.log(df['tempo_emprego'] + 1)

In [38]:
df['idade_tempo'] = df['idade'] * df['tempo_emprego']

In [40]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [42]:
# 7) 
from sklearn.tree import DecisionTreeRegressor

modelo_arvore = DecisionTreeRegressor(
    max_depth=5, 
    random_state=42
)

modelo_arvore.fit(X_train, y_train)

y_pred_arvore = modelo_arvore.predict(X_test)

r2_arvore = r2_score(y_test, y_pred_arvore)

print("R² Árvore:", r2_arvore)

R² Árvore: 0.3711160346488538
